# Empirical Results: Adaptive Boolean Oracle (Marena 2026)

## Cell 1 — Install and clone

In [ ]:
import subprocess, sys, os, shutil

_repo = '/content/quantum_oracle_sketching'
_src  = os.path.join(_repo, 'src')

# ── 1. Wipe repo + any previously installed qos from site-packages ──────
if os.path.isdir(_repo): shutil.rmtree(_repo)
# Uninstall any stale qos egg / dist-info so pip doesn't reuse old paths
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-q', '-y', 'qos',
                'quantum-oracle-sketching', 'quantum_oracle_sketching'],
               capture_output=True)  # ignore errors if not installed
# Remove any stale .pth / egg-link files that could redirect imports
import site
for _sp in site.getsitepackages() + [site.getusersitepackages()]:
    for _f in os.listdir(_sp) if os.path.isdir(_sp) else []:
        if 'qos' in _f.lower() or 'quantum_oracle' in _f.lower() or 'quantum-oracle' in _f.lower():
            _fp = os.path.join(_sp, _f)
            print(f'  removing stale site-packages entry: {_fp}')
            (shutil.rmtree if os.path.isdir(_fp) else os.remove)(_fp)

# ── 2. git clone --depth 1 (fetches live HEAD, bypasses CDN cache) ──────
subprocess.run([
    'git', 'clone', '--depth', '1', '--single-branch', '--branch', 'main',
    'https://github.com/Tommaso-R-Marena/quantum_oracle_sketching.git', _repo
], check=True)
_sha = subprocess.check_output(['git','-C',_repo,'rev-parse','HEAD']).decode().strip()
print(f'✓ Cloned HEAD: {_sha}')

# ── 3. Nuke any committed __pycache__ / .pyc ─────────────────────────────
for _root, _dirs, _files in os.walk(_repo):
    if '__pycache__' in _dirs:
        shutil.rmtree(os.path.join(_root, '__pycache__')); _dirs.remove('__pycache__')
    for _f in _files:
        if _f.endswith('.pyc') or _f.endswith('.pyo'):
            os.remove(os.path.join(_root, _f))
print('✓ __pycache__ cleared')

# ── 4. Install deps only (NO editable install of qos) ───────────────────
# We do NOT run `pip install -e .` here. That creates an editable link
# that can resolve to a stale path from a previous session. Instead we
# prepend src/ to sys.path directly so Python always finds the cloned source.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'jax', 'jaxlib', 'numpy', 'matplotlib',
    'scikit-learn', 'scipy', 'tqdm', 'pyqsp', 'hatchling'], check=True)
print('✓ Deps installed')

# ── 5. Put cloned src/ FIRST on sys.path; purge stale module cache ────────
# Remove any old qos paths that might have crept in
sys.path = [p for p in sys.path if 'qos' not in p.lower() and 'quantum_oracle' not in p.lower()]
sys.path.insert(0, _src)
for _mod in list(sys.modules.keys()):
    if _mod.startswith('qos'): del sys.modules[_mod]

os.chdir(_repo)

# ── 6. Imports ────────────────────────────────────────────────────
from qos.theory.adaptive_lower_bound import (
    compute_bounds, tightness_sweep,
    uniform_vs_adaptive_error_comparison,
    adversarial_sparse_function,
    nisq_adaptive_crossover,
)
from qos.core.oracle_sketch import q_oracle_sketch_boolean, q_oracle_sketch_boolean_adaptive

import jax, jax.numpy as jnp
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib
matplotlib.rcParams.update({
    'font.size': 13, 'axes.titlesize': 13, 'axes.labelsize': 12,
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight'
})
os.makedirs(os.path.join(_repo, 'results'), exist_ok=True)

# ── 7. Version + sanity checks ──────────────────────────────────────────
import qos
print(f'✓ Setup complete. QOS version: {qos.__version__}')

import inspect
_oracle_src_file = inspect.getfile(q_oracle_sketch_boolean_adaptive)
print(f'  oracle_sketch loaded from: {_oracle_src_file}')
# Must be inside the cloned repo
assert _oracle_src_file.startswith(_src), (
    f'WRONG PATH: loaded from {_oracle_src_file}, expected under {_src}')
assert _oracle_src_file.endswith('.py'), f'Loading .pyc not .py: {_oracle_src_file}'

with open(_oracle_src_file) as _fh:
    _raw_src = _fh.read()
# Debug: show first occurrence of 'lax' to confirm it's only in comments
_lax_idx = _raw_src.find('lax.cond')
if _lax_idx >= 0:
    print(f'  FOUND lax.cond at char {_lax_idx}:')
    print(f'  context: {repr(_raw_src[max(0,_lax_idx-80):_lax_idx+40])}')
assert 'lax.cond' not in _raw_src, 'ERROR: lax.cond still in oracle_sketch.py on disk!'
print('✓ Sanity: lax.cond absent from oracle_sketch.py.')
assert 'if pilot_samples > 0 and support_sum > 0:' in _raw_src, 'ERROR: Python if/else not found!'
print('✓ Sanity: Python if/else present.')

## Cell 2 — Figure 1: Adaptive vs Uniform Error

In [ ]:
sample_counts = [200, 500, 1000, 2000, 5000, 10000, 20000]

Na, Ka = 512, 16
print(f'Panel A: N={Na}, K={Ka}, N/K={Na//Ka}, (N/K)^2={(Na//Ka)**2}')
print(f'Adaptive crossover: M ~ K*pi^2/eps^2 = {int(Ka*3.14159**2/0.3**2)} (eps=0.3)')
results_a = uniform_vs_adaptive_error_comparison(N=Na, K=Ka, sample_counts=sample_counts, num_trials=20, key=jax.random.PRNGKey(0))
df1a = pd.DataFrame(results_a)
df1a.to_csv('results/fig1a_N512_K16.csv', index=False)
print(df1a.to_string(index=False))
print(f'N/K improvement factor: {Na/Ka:.0f}x (blind-oracle model)\n')

Nb, Kb = 1024, 16
print(f'Panel B: N={Nb}, K={Kb}, N/K={Nb//Kb}, (N/K)^2={(Nb//Kb)**2}')
results_b = uniform_vs_adaptive_error_comparison(N=Nb, K=Kb, sample_counts=sample_counts, num_trials=20, key=jax.random.PRNGKey(1))
df1b = pd.DataFrame(results_b)
df1b.to_csv('results/fig1b_N1024_K16.csv', index=False)
print(df1b.to_string(index=False))
print(f'N/K improvement factor: {Nb/Kb:.0f}x (blind-oracle model)\n')

M_arr = np.array(sample_counts, float)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
for ax, df, N, K, label in [
    (ax1, df1a, Na, Ka, '(a) N=512, K=16  [N/K=32]'),
    (ax2, df1b, Nb, Kb, '(b) N=1024, K=16  [N/K=64]'),
]:
    pf = np.minimum(0.1, 20.0*K/M_arr); Mm = M_arr*(1-pf)
    ax.errorbar(df['M'], df['uniform_error'],  yerr=df['uniform_std'],  fmt='o-', color='#d62728', label=f'Uniform (N={N})', capsize=4)
    ax.errorbar(df['M'], df['adaptive_error'], yerr=df['adaptive_std'], fmt='s-', color='#1f77b4', label=f'Adaptive (K={K})', capsize=4)
    ax.plot(M_arr, np.sqrt(N*np.pi**2/M_arr),  'r--', alpha=0.5, label=r'$\sqrt{N\pi^2/M}$')
    ax.plot(M_arr, np.sqrt(K*np.pi**2/Mm), 'b--', alpha=0.5, label=r'$\sqrt{K\pi^2/M_{\rm main}}$')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('M'); ax.set_ylabel(r'$L_\infty$ error on supp$(f)$')
    ax.set_title(label); ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
fig.suptitle('Adaptive vs Uniform Oracle Sketch Error (blind-oracle model, 20 trials)', fontsize=13, y=1.02)
plt.tight_layout(); plt.savefig('results/fig1_combined.pdf'); plt.savefig('results/fig1_combined.png'); plt.show()
for df, N, K, fname in [(df1a, Na, Ka, 'fig1a_main'), (df1b, Nb, Kb, 'fig1b_largeN')]:
    M2=np.array(sample_counts,float); pf2=np.minimum(0.1,20.0*K/M2); Mm2=M2*(1-pf2)
    fig2, ax = plt.subplots(figsize=(6,5))
    ax.errorbar(df['M'], df['uniform_error'],  yerr=df['uniform_std'],  fmt='o-', color='#d62728', label=f'Uniform (N={N})', capsize=4)
    ax.errorbar(df['M'], df['adaptive_error'], yerr=df['adaptive_std'], fmt='s-', color='#1f77b4', label=f'Adaptive (K={K})', capsize=4)
    ax.plot(M2, np.sqrt(N*np.pi**2/M2),  'r--', alpha=0.5, label=r'$\sqrt{N\pi^2/M}$')
    ax.plot(M2, np.sqrt(K*np.pi**2/Mm2), 'b--', alpha=0.5, label=r'$\sqrt{K\pi^2/M_{\rm main}}$')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('M'); ax.set_ylabel(r'$L_\infty$ error on supp$(f)$')
    ax.set_title(f'N={N}, K={K}, sparsity={K/N:.1%}'); ax.legend(fontsize=10); ax.grid(True,alpha=0.3)
    plt.tight_layout(); plt.savefig(f'results/{fname}.pdf'); plt.savefig(f'results/{fname}.png'); plt.close()
    print(f'Saved {fname}.pdf / .png')

## Cell 3 — Figure 2: Tightness Sweep

In [ ]:
NK_pairs = [(128,1),(128,6),(128,13),(128,32),(256,3),(256,13),(256,26),(256,64),
            (512,5),(512,26),(512,51),(512,128),(1024,10),(1024,51),(1024,102),(1024,256)]
rows = []
for N, K in NK_pairs:
    r = compute_bounds(N, K, 0.1)
    rows.append({'N':N,'K':K,'sparsity_pct':f'{K/N:.0%}','M_lower':r.M_lower_bound,
                 'M_adaptive':r.M_adaptive_upper,'M_uniform':r.M_uniform_upper,
                 'improvement_factor':r.improvement_factor,'tight':r.is_tight})
df2 = pd.DataFrame(rows)
df2.to_csv('results/fig2_tightness_sweep.csv', index=False); print(df2.to_string(index=False))
pivot_data = df2.pivot_table(index='N', columns='K', values='improvement_factor', aggfunc='first')
fig, ax = plt.subplots(figsize=(9,5))
im = ax.imshow(pivot_data.values, aspect='auto', cmap='Blues')
ax.set_xticks(range(len(pivot_data.columns))); ax.set_xticklabels([f'K={k}' for k in pivot_data.columns])
ax.set_yticks(range(len(pivot_data.index))); ax.set_yticklabels(pivot_data.index)
ax.set_xlabel('K'); ax.set_ylabel('N'); ax.set_title('Improvement Factor N/K (blind-oracle model)')
for i in range(pivot_data.values.shape[0]):
    for j in range(pivot_data.values.shape[1]):
        v = pivot_data.values[i,j]
        if not np.isnan(v): ax.text(j, i, f'{v:.0f}x', ha='center', va='center', fontsize=10)
plt.colorbar(im, ax=ax, label='N/K'); plt.tight_layout()
plt.savefig('results/fig2_tightness_sweep.pdf'); plt.savefig('results/fig2_tightness_sweep.png'); plt.show()

## Cell 4 — Figure 3: NISQ Noise Crossover

In [ ]:
from qos.primitives.noise_model import DepolarizingChannel, crossover_sample_count
N, K, circuit_depth, epsilon_target = 512, 16, 50, 0.3
noise_rates = np.logspace(-4, -1, 30); rows = []
for p in noise_rates:
    r = nisq_adaptive_crossover(N, K, float(p), circuit_depth, epsilon_target)
    rows.append({'noise_rate':p,'epsilon_noise':r['epsilon_noise'],
                 'epsilon_sketch_budget':r['epsilon_sketch_budget'],
                 'M_adaptive':r['M_adaptive_crossover'],'M_uniform':r['M_uniform_crossover'],
                 'beneficial':r['adaptive_still_beneficial']})
df3 = pd.DataFrame(rows); df3.to_csv('results/fig3_nisq_crossover.csv', index=False)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,4))
ax1.loglog(df3['noise_rate'], df3['M_adaptive'], 'b-', label=f'Adaptive (K={K})')
ax1.loglog(df3['noise_rate'], df3['M_uniform'],  'r-', label=f'Uniform (N={N})')
ax1.set_xlabel('p'); ax1.set_ylabel('M*'); ax1.set_title('NISQ Crossover'); ax1.legend(); ax1.grid(True,alpha=0.3)
ax2.semilogx(df3['noise_rate'], df3['epsilon_sketch_budget'], 'g-')
ax2.axhline(y=0, color='k', linestyle='--', alpha=0.5)
ax2.set_xlabel('p'); ax2.set_ylabel(r'$\varepsilon_{\rm sketch}$'); ax2.set_title(f'Sketch Budget (depth={circuit_depth})'); ax2.grid(True,alpha=0.3)
plt.tight_layout(); plt.savefig('results/fig3_nisq_crossover.pdf'); plt.savefig('results/fig3_nisq_crossover.png'); plt.show()

## Cell 5 — Figure 4: k-Forrelation

In [ ]:
try:
    from qos.data.generation import k_forrelation_data; _ok=True
except ImportError as e: print(f'Skip Fig4: {e}'); _ok=False
if _ok:
    rows=[]
    for k in [2,3,4,5]:
        errs=[]
        for t in range(10):
            gen=k_forrelation_data(n=8,k=k,key=jax.random.PRNGKey(t*100+k))
            funcs=gen.sample_functions(jax.random.PRNGKey(t*100+k+1))
            exact=gen.compute_exact_forrelation(funcs)
            truth=((1.0-jnp.sign(funcs[-1]))/2.0).astype(jnp.int32)
            diag,_=q_oracle_sketch_boolean(truth,5000)
            errs.append(abs(float(gen.quantum_query_algorithm(diag))-float(exact)))
        rows.append({'k':k,'mean_error':float(np.mean(errs)),'std_error':float(np.std(errs)),'classical_bound':gen.classical_streaming_complexity(0.1)})
    df4=pd.DataFrame(rows); df4.to_csv('results/fig4_k_forrelation.csv',index=False); print(df4.to_string(index=False))

## Cell 6 — Figure 5: Kernel vs Linear Shadow

In [ ]:
try:
    from qos.core.state_sketch import q_state_sketch_flat, fit_kernel_svm_from_states, q_interferometric_kernel_shadow; _ok=True
except ImportError as e: print(f'Skip Fig5: {e}'); _ok=False
if _ok:
    def _shadow(d,n_tr,n_te,M,key):
        k1,k2,_=jax.random.split(key,3)
        Xtr=jax.random.normal(k1,(n_tr,d)); ytr=jnp.sign(Xtr[:,0]).astype(jnp.int32)
        Xte=jax.random.normal(k2,(n_te,d)); yte=jnp.sign(Xte[:,0]).astype(jnp.int32)
        str_=jnp.stack([q_state_sketch_flat(Xtr[i],M)[0] for i in range(n_tr)])
        ste_=jnp.stack([q_state_sketch_flat(Xte[i],M)[0] for i in range(n_te)])
        alpha=fit_kernel_svm_from_states(str_,ytr)
        kp=jnp.array([q_interferometric_kernel_shadow(str_,ytr,alpha,ste_[i]) for i in range(n_te)])
        lp=ytr[jnp.argmax(jnp.abs(jnp.einsum('id,jd->ij',ste_,str_))**2,axis=1)]
        return float(jnp.mean(kp==yte)), float(jnp.mean(lp==yte))
    rows=[]
    for d in [16,32,64,128]:
        ka_,la_=[],[]
        for t in range(8): ka,la=_shadow(d,20,40,2000,jax.random.PRNGKey(t*7+d)); ka_.append(ka); la_.append(la)
        rows.append({'dim':d,'kernel_acc':np.mean(ka_),'kernel_std':np.std(ka_),'linear_acc':np.mean(la_),'linear_std':np.std(la_)})
    df5=pd.DataFrame(rows); df5.to_csv('results/fig5_kernel_vs_linear.csv',index=False); print(df5.to_string(index=False))

## Cell 7 — Download all results

In [ ]:
results_dir = os.path.join(_repo, 'results')
for fname in sorted(os.listdir(results_dir)):
    print(f'  {fname:45s} {os.path.getsize(os.path.join(results_dir,fname)):>8,} bytes')
shutil.make_archive('/content/marena2026_results', 'zip', results_dir)
print('✓ Download marena2026_results.zip from the Files panel.')